# CIFAR-100 QMobileNetV2 ACS Replication (Colab)

This notebook starts implementation for replicating the repository experiment on CIFAR-100 with QMobileNetV2 and ACS.

Scope:
- Keep hyperparameters exactly as in the repository recipe.
- Run weight-only quantization with bitwidths 2, 3, and 4.
- Change only `--bitwidth` across runs.

Note:
- 2-bit is the documented MobileNetV2 setting in the repository README.
- 3-bit and 4-bit are strict extensions of the same command template.

## 1. Set Up the Project Environment
Import required libraries, verify dependencies, and prepare the working environment for implementation.

In [ ]:
import os
import re
import shlex
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

print('Python environment ready.')

In [ ]:
# Colab-only: mount Drive for persistent logs and outputs
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted at /content/drive')
except Exception as e:
    print('Drive mount skipped or unavailable:', e)

## 2. Create the Initial Project Structure
Define the main folders, files, and module layout needed for the first implementation pass.

In [ ]:
# Edit these two variables before running in Colab
WORK_ROOT = '/content'
REPO_URL = 'https://github.com/<owner>/<repo>.git'
REPO_DIR = f'{WORK_ROOT}/Supplementary_TMLR_Robust_and_Efficient_Quantization_aware_Training_via_Coreset_Selection'
DRIVE_OUT = '/content/drive/MyDrive/acs_qat_runs'
DATA_DIR = '/content/data'
BITWIDTHS = [2, 3, 4]

os.makedirs(DRIVE_OUT, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print('WORK_ROOT:', WORK_ROOT)
print('REPO_DIR :', REPO_DIR)
print('DRIVE_OUT:', DRIVE_OUT)
print('DATA_DIR :', DATA_DIR)
print('BITWIDTHS:', BITWIDTHS)

In [ ]:
# Clone the repository only if missing, then enter it
if not os.path.isdir(REPO_DIR):
    if '<owner>' in REPO_URL or '<repo>' in REPO_URL:
        raise ValueError('Please set REPO_URL to the actual repository URL before running this cell.')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print('Repository already exists:', REPO_DIR)

os.chdir(REPO_DIR)
print('Current working directory:', os.getcwd())

In [ ]:
# Install dependencies from requirements.txt and print torch versions
req_file = Path('requirements.txt')
if req_file.exists():
    subprocess.run(['pip', 'install', '-r', str(req_file)], check=True)
else:
    print('requirements.txt not found, skipping pip install.')

import torch
import torchvision
print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)

## 3. Implement the Core Module
Write the first version of the main functions or classes that provide the core application behavior.

In [ ]:
# Checkpoint path guard for pretrained model naming mismatch
src_ckpt = Path('pretrained_models/CIFAR100_Mobilenetv2_72.56.ckpt')
dst_dir = Path('pretrained_model')
dst_ckpt = dst_dir / 'CIFAR100_Mobilenetv2_72.56.ckpt'

if Path('pretrained_models').is_dir() and not dst_dir.is_dir():
    dst_dir.mkdir(parents=True, exist_ok=True)

if src_ckpt.exists() and not dst_ckpt.exists():
    shutil.copy2(src_ckpt, dst_ckpt)
    print('Copied checkpoint to expected path:', dst_ckpt)

if not dst_ckpt.exists():
    raise FileNotFoundError(
        'Checkpoint missing at ./pretrained_model/CIFAR100_Mobilenetv2_72.56.ckpt. '
        'Please place it there before running training.'
    )

print('Checkpoint ready:', dst_ckpt.resolve())

In [ ]:
def build_cmd(bitwidth: int) -> str:
    log_path = f"{DRIVE_OUT}/lsq_mobilenetv2_cifar100_ACS10_bit{bitwidth}.log"
    cmd = (
        'python main.py '
        '--fraction 0.1 '
        '--dataset CIFAR100 '
        '--data_path ' + shlex.quote(DATA_DIR) + ' '
        '--model QMobilenetv2 '
        '--selection ACS '
        '--num_exp 5 '
        '--epochs 200 '
        '--min_lr 0 '
        '--lr 0.01 '
        '--weight_decay 5e-4 '
        '--batch-size 256 '
        '--scheduler LambdaLR '
        '--resume ./pretrained_model/CIFAR100_Mobilenetv2_72.56.ckpt '
        f'--bitwidth {bitwidth} '
        '--log ' + shlex.quote(log_path)
    )
    return cmd

## 4. Add Configuration and Constants
Define reusable constants, settings, and configuration values used by the implementation.

In [ ]:
# Fixed hyperparameters from the repository CIFAR100 MobileNetV2 command
FIXED_ARGS = {
    'fraction': 0.1,
    'dataset': 'CIFAR100',
    'model': 'QMobilenetv2',
    'selection': 'ACS',
    'num_exp': 5,
    'epochs': 200,
    'min_lr': 0,
    'lr': 0.01,
    'weight_decay': 5e-4,
    'batch_size': 256,
    'scheduler': 'LambdaLR',
    'resume': './pretrained_model/CIFAR100_Mobilenetv2_72.56.ckpt',
}

print(FIXED_ARGS)

## 5. Write Basic Unit Tests
Create initial tests for the core functions or classes to validate expected behavior.

In [ ]:
# Basic checks: only bitwidth changes between generated commands
cmds = {bw: build_cmd(bw) for bw in BITWIDTHS}

for bw, cmd in cmds.items():
    assert f'--bitwidth {bw}' in cmd
    assert '--dataset CIFAR100' in cmd
    assert '--model QMobilenetv2' in cmd
    assert '--selection ACS' in cmd

normalized = []
for bw in BITWIDTHS:
    normalized.append(re.sub(r'--bitwidth\s+\d+', '--bitwidth X', cmds[bw]))

assert len(set(normalized)) == 1, 'Commands differ by more than bitwidth.'
print('Basic unit tests passed.')

In [ ]:
# Dry-run: print the exact commands before execution
for bw in BITWIDTHS:
    print(f'\n[bitwidth={bw}]')
    print(build_cmd(bw))

## Persistence Across Session Disconnects
This section stores run progress in Google Drive and skips bitwidths that are already complete.

Behavior:
- A state file is written to Drive after each successful bitwidth run.
- On reconnect, rerunning the execution cell resumes from pending bitwidths.
- Limitation: this repository does not support true mid-epoch resume for this ACS + CIFAR100 setup without code changes, so an interrupted bitwidth restarts from that bitwidth.

In [ ]:
import json
from datetime import datetime

STATE_PATH = Path(DRIVE_OUT) / 'run_state_cifar100_qmobilenetv2_acs_bits234.json'
BEST_ACC_PATTERN = re.compile(r'Best accuracy:\s*([0-9]*\.?[0-9]+)')
EXPECTED_ENTRIES = int(FIXED_ARGS['num_exp'])


def load_state(path: Path) -> dict:
    if path.exists():
        return json.loads(path.read_text())
    return {'created_at': datetime.utcnow().isoformat(), 'runs': {}}


def save_state(path: Path, state: dict) -> None:
    state['updated_at'] = datetime.utcnow().isoformat()
    path.write_text(json.dumps(state, indent=2))


def log_entry_count(log_path: Path) -> int:
    if not log_path.exists():
        return 0
    text = log_path.read_text(errors='ignore')
    return len(BEST_ACC_PATTERN.findall(text))


def bitwidth_log_path(bitwidth: int) -> Path:
    return Path(f"{DRIVE_OUT}/lsq_mobilenetv2_cifar100_ACS10_bit{bitwidth}.log")


state = load_state(STATE_PATH)
print('State file:', STATE_PATH)
print(json.dumps(state, indent=2))

## 6. Run the First Execution Check
Execute the implemented code and tests to confirm the notebook runs correctly in the current environment.

In [ ]:
# Full training execution (long-running): resume from pending bitwidths using Drive state
state = load_state(STATE_PATH)

for bw in BITWIDTHS:
    run_key = str(bw)
    log_path = bitwidth_log_path(bw)
    existing_entries = log_entry_count(log_path)
    already_done = state.get('runs', {}).get(run_key, {}).get('status') == 'completed'

    # Skip if previously marked complete and log already has all expected experiment entries.
    if already_done and existing_entries >= EXPECTED_ENTRIES:
        print(f'\n=== Skipping bitwidth {bw}: already completed ({existing_entries} entries) ===')
        continue

    cmd = build_cmd(bw)
    state.setdefault('runs', {})[run_key] = {
        'status': 'running',
        'started_at': datetime.utcnow().isoformat(),
        'log_path': str(log_path),
        'existing_entries_before_run': existing_entries,
        'cmd': cmd,
    }
    save_state(STATE_PATH, state)

    print(f'\n=== Running bitwidth {bw} ===')
    print(cmd)
    result = subprocess.run(cmd, shell=True)

    if result.returncode != 0:
        state['runs'][run_key]['status'] = 'failed'
        state['runs'][run_key]['returncode'] = int(result.returncode)
        state['runs'][run_key]['finished_at'] = datetime.utcnow().isoformat()
        state['runs'][run_key]['entries_after_run'] = log_entry_count(log_path)
        save_state(STATE_PATH, state)
        raise RuntimeError(f'Run failed for bitwidth {bw} with return code {result.returncode}')

    state['runs'][run_key]['status'] = 'completed'
    state['runs'][run_key]['returncode'] = 0
    state['runs'][run_key]['finished_at'] = datetime.utcnow().isoformat()
    state['runs'][run_key]['entries_after_run'] = log_entry_count(log_path)
    save_state(STATE_PATH, state)

print('\nAll requested runs are completed or already done.')
print('Final state saved at:', STATE_PATH)

In [ ]:
# Parse logs and summarize Best accuracy values
rows = []
pattern = re.compile(r'Best accuracy:\s*([0-9]*\.?[0-9]+)')

for bw in BITWIDTHS:
    log_path = Path(f"{DRIVE_OUT}/lsq_mobilenetv2_cifar100_ACS10_bit{bw}.log")
    vals = []
    if log_path.exists():
        text = log_path.read_text(errors='ignore')
        vals = [float(x) for x in pattern.findall(text)]

    rows.append({
        'bitwidth': bw,
        'num_entries': len(vals),
        'mean_best_acc': float(np.mean(vals)) if vals else np.nan,
        'std_best_acc': float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan,
        'max_best_acc': float(np.max(vals)) if vals else np.nan,
        'log_path': str(log_path),
    })

summary_df = pd.DataFrame(rows)
summary_df

In [ ]:
# Save summary table to Drive
summary_csv = Path(DRIVE_OUT) / 'summary_cifar100_qmobilenetv2_acs_bits234.csv'
summary_df.to_csv(summary_csv, index=False)
print('Saved summary to:', summary_csv)
summary_df

### Notes
- This setup keeps hyperparameters fixed and only changes bitwidth (2, 3, 4).
- In this repository, QMobileNetV2 uses LSQ quantized convolution layers for weights; activations remain standard ReLU6 in the model blocks.
- For strict reproducibility of run-to-run randomness, you may add `--seed` explicitly, but omitting it follows repository default behavior.